In [116]:
import pandas as pd
import numpy as np

In [117]:
flood_area_df=pd.read_csv('../../../data/processed/flood_area.csv')
flood_dis_df=pd.read_csv('../../../data/processed/Flooding_Disaster.csv')
riv_df=pd.read_csv('../../../data/processed/riv_pro_df.csv')
solid_df=pd.read_csv('../../../data/processed/solid_pump_data.csv')
weather_df=pd.read_csv('../../../data/processed/weather_rain_data.csv')

In [118]:
# 날짜 데이터 없는 df부터 병합
merged_df=pd.merge(riv_df, solid_df, how='left', on='SIGUNGU')
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1467 entries, 0 to 1466
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   RIV_GRD       1467 non-null   int64  
 1   RIV_DIS_MIN   1467 non-null   float64
 2   RIV_DIS_GRD   1467 non-null   int64  
 3   SIGUNGU       1467 non-null   int64  
 4   DRAINAGE_GRD  254 non-null    float64
 5   PUMP_CNT      254 non-null    float64
dtypes: float64(3), int64(3)
memory usage: 68.9 KB


In [119]:
# 중복값 처리
flood_dis_df.duplicated().sum()
flood_dis_df = flood_dis_df.drop_duplicates().reset_index(drop=True)
flood_dis_df

,FLOOD_YEAR,FLOOD_GRD,SIGUNGU,FLOOD_MONTH,DAM_RAIN,DEAD
0,2017,3,43111,7,188.0,0.0
1,2017,2,43111,7,188.0,0.0
2,2017,3,43114,7,188.0,0.0
3,2017,1,43114,7,188.0,0.0
4,2017,2,43114,7,188.0,0.0
...,...,...,...,...,...,...
814,2022,1,11170,8,188.0,0.0
815,2022,1,11410,9,188.0,0.0
816,2021,2,45710,7,188.0,0.0
817,2021,1,45720,7,188.0,0.0


In [120]:
# 중복값 처리
weather_df.info()
weather_df.duplicated().sum()
weather_df = weather_df.drop_duplicates().reset_index(drop=True)
weather_df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14960 entries, 0 to 14959
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   SIGUNGU      14960 non-null  int64 
 1   FLOOD_YEAR   14960 non-null  int64 
 2   FLOOD_MONTH  14960 non-null  int64 
 3   RAIN_TOTAL   14960 non-null  object
dtypes: int64(3), object(1)
memory usage: 467.6+ KB


np.int64(0)

In [121]:
# 중복 지역, 연월일 다수 FLOOD AREA 최대값으로 중복값 처리 / 최대 이만큼 침수될 수 있음
flood_area_result = flood_area_df.sort_values('FLOOD_AREA', ascending=False).drop_duplicates(subset=['SIGUNGU','FLOOD_YEAR','FLOOD_MONTH'], keep='first')
print(flood_area_result)

       SIGUNGU  FLOOD_YEAR  FLOOD_MONTH  FLOOD_AREA
6755     48270        2020            9   3229418.8
6949     46870        2020            8   2658469.0
5857     46170        2020            8   2454810.2
5191     45190        2020            8   2130538.2
3055     48270        2019           10   2053835.8
...        ...         ...          ...         ...
13567    41390        2022            6        88.9
3954     41150        2020            7        68.0
7898     41210        2021            7        63.3
12195    44230        2021            7        48.6
8840     28245        2021            8        45.6

[321 rows x 4 columns]


In [122]:
merged_df_2=pd.merge(flood_area_result, weather_df, how='outer', on=['SIGUNGU','FLOOD_YEAR','FLOOD_MONTH'])
merged_df_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11315 entries, 0 to 11314
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SIGUNGU      11315 non-null  int64  
 1   FLOOD_YEAR   11315 non-null  int64  
 2   FLOOD_MONTH  11315 non-null  int64  
 3   FLOOD_AREA   342 non-null    float64
 4   RAIN_TOTAL   11083 non-null  object 
dtypes: float64(1), int64(3), object(1)
memory usage: 442.1+ KB


In [123]:
merged_df_3=pd.merge(merged_df_2, flood_dis_df, how='left', on=['SIGUNGU','FLOOD_YEAR','FLOOD_MONTH'])
merged_df_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11859 entries, 0 to 11858
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SIGUNGU      11859 non-null  int64  
 1   FLOOD_YEAR   11859 non-null  int64  
 2   FLOOD_MONTH  11859 non-null  int64  
 3   FLOOD_AREA   886 non-null    float64
 4   RAIN_TOTAL   11318 non-null  object 
 5   FLOOD_GRD    886 non-null    float64
 6   DAM_RAIN     886 non-null    float64
 7   DEAD         886 non-null    float64
dtypes: float64(4), int64(3), object(1)
memory usage: 741.3+ KB


In [124]:
final_df = pd.merge(merged_df_3, merged_df, how='left', on='SIGUNGU')
final_df

,SIGUNGU,FLOOD_YEAR,FLOOD_MONTH,FLOOD_AREA,RAIN_TOTAL,FLOOD_GRD,DAM_RAIN,DEAD,RIV_GRD,RIV_DIS_MIN,RIV_DIS_GRD,DRAINAGE_GRD,PUMP_CNT
0,11110,2016,1,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11110,2016,2,NaN,47.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11110,2016,3,NaN,40.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11110,2016,4,NaN,76.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11110,2016,5,NaN,160.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
162399,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,470.451261,3.0,NaN,NaN
162400,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,484.734599,3.0,NaN,NaN
162401,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,475.856789,3.0,NaN,NaN
162402,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,570.475966,3.0,NaN,NaN


In [125]:
final_df[final_df['FLOOD_AREA'].isnull()]

,SIGUNGU,FLOOD_YEAR,FLOOD_MONTH,FLOOD_AREA,RAIN_TOTAL,FLOOD_GRD,DAM_RAIN,DEAD,RIV_GRD,RIV_DIS_MIN,RIV_DIS_GRD,DRAINAGE_GRD,PUMP_CNT
0,11110,2016,1,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11110,2016,2,NaN,47.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11110,2016,3,NaN,40.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11110,2016,4,NaN,76.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11110,2016,5,NaN,160.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
162399,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,470.451261,3.0,NaN,NaN
162400,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,484.734599,3.0,NaN,NaN
162401,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,475.856789,3.0,NaN,NaN
162402,52800,2025,11,NaN,21.5,NaN,NaN,NaN,3.0,570.475966,3.0,NaN,NaN


In [126]:
final_df['FLOOD_AREA'] = final_df['FLOOD_AREA'].fillna(0)
final_df['FLOOD_GRD'] = final_df['FLOOD_GRD'].fillna(0)
final_df['DAM_RAIN'] = final_df['DAM_RAIN'].fillna(0)
final_df['DEAD'] = final_df['DEAD'].fillna(0)
final_df['RIV_GRD'] = final_df['RIV_GRD'].fillna(0)
final_df['RIV_DIS_MIN'] = final_df['RIV_DIS_MIN'].fillna(0)
final_df['RIV_DIS_GRD'] = final_df['RIV_DIS_GRD'].fillna(0)
final_df['DRAINAGE_GRD'] = final_df['DRAINAGE_GRD'].fillna(0)
final_df['PUMP_CNT'] = final_df['PUMP_CNT'].fillna(0)

In [127]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162404 entries, 0 to 162403
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   SIGUNGU       162404 non-null  int64  
 1   FLOOD_YEAR    162404 non-null  int64  
 2   FLOOD_MONTH   162404 non-null  int64  
 3   FLOOD_AREA    162404 non-null  float64
 4   RAIN_TOTAL    160550 non-null  object 
 5   FLOOD_GRD     162404 non-null  float64
 6   DAM_RAIN      162404 non-null  float64
 7   DEAD          162404 non-null  float64
 8   RIV_GRD       162404 non-null  float64
 9   RIV_DIS_MIN   162404 non-null  float64
 10  RIV_DIS_GRD   162404 non-null  float64
 11  DRAINAGE_GRD  162404 non-null  float64
 12  PUMP_CNT      162404 non-null  float64
dtypes: float64(9), int64(3), object(1)
memory usage: 16.1+ MB


In [128]:
total_data = final_df.copy()

#  CSV 저장
total_data.to_csv("../../../data/processed/final_data.csv", index=False, encoding="utf-8-sig")
print(" 저장 완료: ../../../data/processed/final_data.csv")

 저장 완료: ../../../data/processed/final_data.csv
